# Merge the LoRA adapter into the base model (Kaggle)

Folds the adapter into `Qwen2.5-Coder-1.5B-Instruct`: every adapted layer becomes `W + B·A`, saved
as one standalone model. Output works without PEFT, and is the input for `quantize_gguf_Kaggle.ipynb`
(and what you'd upload to Hugging Face).

Output, in the version's Output tab: `merged-model/`, the merged model in bf16 (~3.1GB),
Hugging Face format, with its tokenizer.

Before saving, it checks that the merged model gives the same output as base + adapter.

**Setup on Kaggle:**
1. Settings -> Internet -> On. No accelerator needed: merging runs on CPU (~8GB RAM; Kaggle has 30GB).
2. Add Data -> attach the adapter dataset (`adapter_config.json` + `adapter_model.safetensors`)
3. Run with **Save Version -> Save & Run All (Commit)**

In [ ]:
# --- Dependencies ---
# Same torchao pin as the fine-tuning notebook.
%pip install -q -U transformers peft accelerate "torchao>=0.16.0"

In [ ]:
import glob
import os
import shutil

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Config ---
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
OUT_DIR = "/kaggle/working/merged-model"
ADAPTER_DIR = "/tmp/adapter"

def find_input(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Couldn't find {filename} anywhere under /kaggle/input. Check Add Data attached it. "
            f"Found: {glob.glob('/kaggle/input/**/*', recursive=True)[:50]}"
        )
    return matches[0]


# PEFT needs the adapter's config and weights in one folder; Kaggle datasets may split them.
os.makedirs(ADAPTER_DIR, exist_ok=True)
for filename in ["adapter_config.json", "adapter_model.safetensors"]:
    shutil.copy(find_input(filename), ADAPTER_DIR)


# --- Merge ---
# In fp32, so W + B·A is computed without rounding; stored as bf16 (Qwen's native dtype) afterwards.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()

probe = tokenizer(
    "Write a Conventional Commits message for this diff.\n\n"
    "diff --git a/app.py b/app.py\n-    return x\n+    return x or default\n\nCommit message:",
    return_tensors="pt",
)
with torch.no_grad():
    logits_adapter = model(**probe).logits

model = model.merge_and_unload()
with torch.no_grad():
    logits_merged = model(**probe).logits

# Merging is exact math, so only float rounding should differ.
max_diff = (logits_adapter - logits_merged).abs().max().item()
print(f"Max logit difference, base + adapter vs merged: {max_diff:.2e}")
assert max_diff < 1e-2, "Merged model doesn't match base + adapter; not saving it"


# --- Save ---
model.to(torch.bfloat16).save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
total_gb = sum(os.path.getsize(f"{OUT_DIR}/{f}") for f in os.listdir(OUT_DIR)) / 1e9
print(f"Saved merged model to {OUT_DIR} ({total_gb:.2f} GB):", sorted(os.listdir(OUT_DIR)))